# Assignment week 3 working with the cloud

Today we will be working with computing and data in the cloud. We will work with data from our world in data (https://ourworldindata.org/grapher/cattle-livestock-count-heads?tab=table) about the number of cattle per country and over time. We will use this to answer some questions about how the number of cattle has changed over time and how this differs per country. We will also work with the GitHub integration for version control. Create your own compute to run this notebook and fill in the questions using markdown cells. When you are finished, push the finished notebook to your personal GitHub and hand in the link on Canvas.

# Import the data

In [0]:
import pandas as pd
import requests

# Fetch the data.
cowstats = pd.read_csv("https://ourworldindata.org/grapher/cattle-livestock-count-heads.csv?v=1&csvType=full&useColumnShortNames=false", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/cattle-livestock-count-heads.metadata.json?v=1&csvType=full&useColumnShortNames=false").json()

In [0]:
#Check whether 
print(cowstats.head())

## Question 1 

1.1. Where is this data stored?
 
1.2 How is databricks able to load it in? - 

1.3 What are the advantages of using this way of loading the data compared to downloading it on your computer? 


Fill in your answers in this cell

1.1. "https://ourworldindata.org/grapher/cattle-livestock-count-heads.metadata.json?v=1&csvType=full&useColumnShortNames=false"
1.2 cowstats = pd.read_csv
1.3 This saves storage on your computer and allows for the amount of information to scale

# Inspecting your data 

In [0]:
#Databricks has a very nice display function to inspect your data that also allows you to make data summaries and simple visualizations. Let's try it out!

#first display the data 
display(cowstats)

# Press the plus sign to create a data profile
# Give your data profile a name other then Data Profile 1

Databricks data profile. Run in Databricks to view.

In [0]:
# Comprehensive summary of the cattle dataset
print("=" * 60)
print("CATTLE LIVESTOCK DATA SUMMARY")
print("=" * 60)

# Basic dataset info
print(f"\nTotal records: {len(cowstats):,}")
print(f"Number of unique countries/regions: {cowstats['Entity'].nunique()}")
print(f"Time period: {cowstats['Year'].min()} to {cowstats['Year'].max()}")

# Cattle statistics
print("\n" + "=" * 60)
print("CATTLE COUNT STATISTICS (animals)")
print("=" * 60)
print(f"Mean: {cowstats['Cattle - Stocks (animals)'].mean():,.0f}")
print(f"Median: {cowstats['Cattle - Stocks (animals)'].median():,.0f}")
print(f"Minimum: {cowstats['Cattle - Stocks (animals)'].min():,.0f}")
print(f"Maximum: {cowstats['Cattle - Stocks (animals)'].max():,.0f}")
print(f"Standard deviation: {cowstats['Cattle - Stocks (animals)'].std():,.0f}")

# Show data types
print("\n" + "=" * 60)
print("COLUMN INFORMATION")
print("=" * 60)
print(cowstats.dtypes)

# Check for missing values
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)
print(cowstats.isnull().sum())

## Question 2 

2.1 How many countries are in this dataset? 

2.2 What is the time period that data was collected over? 

2.3 What is the mean number of cows per country? 

2.4. What is the median number of cows per country? 

2.5 Why is there such a big difference between the mean and median number of cows?

Create your own markdown cell below to fill in your answers.

2.1 There are 248 countires included

2.2 The time period was 1961 - 2024

2.3 Mean: 38.4 million animals per record

2.4 Median: 2.1 million animals per record

2.5 The huge difference between the mean (38.4M) and median (2.1M) indicates the dataset includes both small countries with few cattle and large countries or regional aggregates (like India, China, or continental totals) with massive cattle populations. This creates a right-skewed distribution.


# Create a visualisation

Again use the display function, but this time click on the plus sign and make a graph. 

Create an easy to read plot with the number of cows over time. Put the years on the x-axis and the number of cows on the y-axis. Change the labels on the axis so it is clear what they mean and have the appropriate unit. 

Create a new graph but this time group the data per country to get one line per country.

Create a filter to see the change in cows in North America. 

Change the filter to see the change of number of cows in Africa but only starting from the year 2000.

In [0]:
import matplotlib.pyplot as plt

# Filter for World data to show overall trend
world_data = cowstats[cowstats['Entity'] == 'World'].sort_values('Year')

# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(world_data['Year'], world_data['Cattle - Stocks (animals)'], linewidth=2, color='#2E86AB')
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Cattle (millions of animals)', fontsize=12, fontweight='bold')
plt.title('Global Cattle Population Over Time (1961-2024)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Format y-axis to show values in millions
ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.0f}'))

plt.tight_layout()
plt.show()

print(f"\nGlobal cattle population in 1961: {world_data.iloc[0]['Cattle - Stocks (animals)']:,.0f}")
print(f"Global cattle population in 2024: {world_data.iloc[-1]['Cattle - Stocks (animals)']:,.0f}")

In [0]:
# Get top 10 countries by most recent cattle count (2024)
top_countries = cowstats[cowstats['Year'] == 2024].nlargest(10, 'Cattle - Stocks (animals)')['Entity'].tolist()

# Filter for these countries and remove aggregate regions
countries_to_plot = [c for c in top_countries if c not in ['World', 'Africa', 'Asia', 'Europe', 'Americas', 'Oceania', 'Africa (FAO)', 'Asia (FAO)']]

# Create plot with one line per country
plt.figure(figsize=(14, 8))

for country in countries_to_plot[:8]:  # Plot top 8 individual countries
    country_data = cowstats[cowstats['Entity'] == country].sort_values('Year')
    plt.plot(country_data['Year'], country_data['Cattle - Stocks (animals)'], 
             linewidth=2, label=country, marker='o', markersize=3, alpha=0.8)

plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Cattle (millions of animals)', fontsize=12, fontweight='bold')
plt.title('Cattle Population by Country Over Time (Top 8 Countries)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10, framealpha=0.9)
plt.grid(True, alpha=0.3)

# Format y-axis to show values in millions
ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.0f}'))

plt.tight_layout()
plt.show()

In [0]:
# Filter for North America
north_america_data = cowstats[cowstats['Entity'] == 'North America'].sort_values('Year')

# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(north_america_data['Year'], north_america_data['Cattle - Stocks (animals)'], 
         linewidth=2.5, color='#A23B72', marker='o', markersize=4)
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Cattle (millions of animals)', fontsize=12, fontweight='bold')
plt.title('Cattle Population in North America Over Time (1961-2024)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Format y-axis to show values in millions
ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.0f}'))

plt.tight_layout()
plt.show()

# Show key statistics
print(f"\nNorth America cattle in 1961: {north_america_data.iloc[0]['Cattle - Stocks (animals)']:,.0f}")
print(f"North America cattle in 2024: {north_america_data.iloc[-1]['Cattle - Stocks (animals)']:,.0f}")
print(f"Peak year: {north_america_data.loc[north_america_data['Cattle - Stocks (animals)'].idxmax(), 'Year']:.0f}")
print(f"Peak count: {north_america_data['Cattle - Stocks (animals)'].max():,.0f}")

In [0]:
# Filter for Africa starting from year 2000
africa_data = cowstats[(cowstats['Entity'] == 'Africa') & (cowstats['Year'] >= 2000)].sort_values('Year')

# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(africa_data['Year'], africa_data['Cattle - Stocks (animals)'], 
         linewidth=2.5, color='#F18F01', marker='o', markersize=4)
plt.xlabel('Year', fontsize=12, fontweight='bold')
plt.ylabel('Number of Cattle (millions of animals)', fontsize=12, fontweight='bold')
plt.title('Cattle Population in Africa (2000-2024)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Format y-axis to show values in millions
ax = plt.gca()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.0f}'))

plt.tight_layout()
plt.show()

# Show key statistics
cattle_2000 = africa_data.iloc[0]['Cattle - Stocks (animals)']
cattle_2024 = africa_data.iloc[-1]['Cattle - Stocks (animals)']
increase = cattle_2024 - cattle_2000
percent_increase = (increase / cattle_2000) * 100
multiplier = cattle_2024 / cattle_2000

print(f"\nAfrica cattle in 2000: {cattle_2000:,.0f}")
print(f"Africa cattle in 2024: {cattle_2024:,.0f}")
print(f"Increase: {increase:,.0f} animals")
print(f"Percentage increase: {percent_increase:.1f}%")
print(f"Multiplier (2024 vs 2000): {multiplier:.2f}x")

## Question 3

3.1 How many more cows where there in 2024 compared to 1965? 

3.2 What is the percentage increase in cows 2024 versus 1965. (tip Python is very good at math, if you create a coding cell, you can use that as a calculator)

3.3 What is the trend in the number of cows in North America?

3.4 Since the year 2000, how many times more cows are there in Africa? 

In [0]:
# Question 3 calculations

# 3.1 & 3.2: How many more cows in 2024 vs 1965 (World data)
world_1965 = cowstats[(cowstats['Entity'] == 'World') & (cowstats['Year'] == 1965)]['Cattle - Stocks (animals)'].values[0]
world_2024 = cowstats[(cowstats['Entity'] == 'World') & (cowstats['Year'] == 2024)]['Cattle - Stocks (animals)'].values[0]

increase_1965_2024 = world_2024 - world_1965
percentage_increase = (increase_1965_2024 / world_1965) * 100

print("=" * 60)
print("QUESTION 3.1 & 3.2: Global cattle 1965 vs 2024")
print("=" * 60)
print(f"Cattle in 1965: {world_1965:,.0f}")
print(f"Cattle in 2024: {world_2024:,.0f}")
print(f"\nIncrease: {increase_1965_2024:,.0f} more cattle")
print(f"Percentage increase: {percentage_increase:.1f}%")

# 3.3: Trend in North America
print("\n" + "=" * 60)
print("QUESTION 3.3: North America trend")
print("=" * 60)
print("The North America cattle population shows:")
print("- Started at 139.7M in 1961")
print("- Peaked at 190.0M in 1975")
print("- Declined to 154.4M by 2024")
print("- Overall trend: Increased until mid-1970s, then declined")

# 3.4: Africa multiplier since 2000
print("\n" + "=" * 60)
print("QUESTION 3.4: Africa cattle growth since 2000")
print("=" * 60)
print(f"Africa cattle in 2000: {cattle_2000:,.0f}")
print(f"Africa cattle in 2024: {cattle_2024:,.0f}")
print(f"\nMultiplier: {multiplier:.2f}x (or {multiplier:.2f} times more)")
print(f"That's a {percent_increase:.1f}% increase!")

# Pushing your results 

Push this notebook to your GitHub. Make sure to not push it to the original repo but instead your own forked repo. Hand in the GitHub link to this notebook on Canvas. 